# 04 — Multi-temporal Change Detection

**Goal:** quantify built-up growth in Nairobi by applying the Random Forest classifier
(notebook 03, ~86% held-out accuracy) to two time points and comparing the results — the first
half of the project's actual objective ("built-up area change detection").

**Baseline year:** 2019, dry season (June–September), same seasonal window as the 2024
composite so the comparison isn't confounded by seasonal vegetation differences. 8 scenes pass
the <20% cloud filter for that window (2017's equivalent window only has 3 — too sparse to trust).
A 5-year gap is enough to expect visible urban growth without going back further than the data
comfortably supports.

**This notebook took two failed attempts to get an honest result from, and that failure is worth
recording in detail** — see the "Diagnosing a bad result" section below.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite
from classification import (
    build_feature_image, get_worldcover_builtup, sample_training_points,
    train_random_forest, classify_builtup, normalize_to_reference, FEATURE_NAMES,
)

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()

composite_2024, scenes_2024 = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
composite_2019, scenes_2019 = get_sentinel2_composite(
    nairobi, start_date='2019-06-01', end_date='2019-09-30', cloud_threshold=20
)
print(f'2024 composite: {scenes_2024} scenes')
print(f'2019 composite: {scenes_2019} scenes')

2024 composite: 11 scenes
2019 composite: 8 scenes


## Diagnosing a bad result

The first version of this notebook trained the RF once on 2024 (the only year WorldCover labels
are contemporaneous with) and applied it unchanged to both composites — methodologically the
right call, change detection needs one fixed decision boundary applied consistently across time.
But the raw output failed an obvious sanity check: **32.1% of Nairobi came back as "lost
built-up"** (built-up in 2019, not in 2024) — implausible for a growing city, and far too large
to be real noise.

**First diagnosis:** per-band means across the whole city differed enormously between composites
— e.g. B4 (red) averaged 1770 in 2019 vs. 952 in 2024, a ~90% difference, uniformly across every
band, not just built-up-related ones. That's a systematic brightness shift, not a land-cover
signal. Not the well-known Sentinel-2 baseline-offset bug (already handled by using the
`_HARMONIZED` collection) — direction and magnitude didn't match that bug's signature.

**Second diagnosis, after the first fix only partly worked:** linearly matching each band's
mean/stdDev over the whole city brought "lost built-up" down from 32.1% to ~16%, real progress,
but still too high, and checking band statistics *inside Nairobi National Park* (a large,
protected, presumably near-unchanged reference area) showed why: the 2019 park interior had a
**stdDev roughly 5–10x higher than 2024's** in the visible bands (e.g. B2: 1063 vs. 99) — far
too much internal variance for what should be a spectrally uniform natural area. That pointed at
residual cloud/shadow contamination surviving into the composite, not a simple brightness
offset. The pipeline (`acquisition.py`) only ever filtered by whole-scene
`CLOUDY_PIXEL_PERCENTAGE`, with no per-pixel cloud/shadow mask — fine when 11 clean scenes go
into a median (2024), much less fine with only 8 scenes (2019), where a median has fewer
alternatives to outvote a contaminated pixel.

**Fix applied at the source:** `acquisition.py`'s `get_sentinel2_composite` now masks clouds,
cloud shadow, and cirrus per-pixel via the Sentinel-2 SCL band before compositing (see that
file). This is a real correctness fix to the shared, reused module, not a notebook-04-only patch
— notebook 03 was re-run against it and its numbers barely moved (86.2%→85.9% held-out accuracy,
2024's composite was already well-covered), confirming the fix mattered for the sparse year, not
the well-covered one.

**Masking alone wasn't sufficient either** (still ~21% "lost built-up") — there's a genuine
residual radiometric difference between the two dry seasons beyond cloud contamination
(atmospheric conditions, antecedent soil moisture, sensor calibration drift). Masking +
mean/stdDev band normalization together is the combination used below.

## Train once on 2024, normalize 2019 to match, apply consistently

Same training procedure as notebook 03: stratified sample against WorldCover labels, 70/30
train/test split, 100-tree Random Forest, trained only on 2024 (the year the labels are
contemporaneous with). The 2019 composite is linearly normalized to match 2024's per-band
mean/stdDev over Nairobi (`normalize_to_reference`, `src/classification.py`) before either
composite is classified.

In [2]:
composite_2019_norm = normalize_to_reference(composite_2019, composite_2024, nairobi)

features_2024 = build_feature_image(composite_2024)
features_2019 = build_feature_image(composite_2019_norm)

worldcover_builtup = get_worldcover_builtup(nairobi)
train_samples, test_samples = sample_training_points(features_2024, worldcover_builtup, nairobi)
classifier = train_random_forest(train_samples)

test_accuracy = test_samples.classify(classifier).errorMatrix(
    'builtup', 'classification'
).accuracy().getInfo()
print(f'Held-out test accuracy (sanity check vs. notebook 03): {test_accuracy * 100:.1f}%')

Held-out test accuracy (sanity check vs. notebook 03): 85.9%


## Classify both years and compute the change map

- **New built-up** = built-up in 2024 AND NOT built-up in 2019 — the growth signal.
- **Lost built-up** = built-up in 2019 AND NOT in 2024 — expected to be small in a growing city;
  this remains the running sanity check on whether the two composites are actually comparable,
  after everything done above to make them so.

In [3]:
builtup_2019 = classify_builtup(features_2019, classifier)
builtup_2024 = classify_builtup(features_2024, classifier)

new_builtup = builtup_2024.And(builtup_2019.Not()).rename('change')
lost_builtup = builtup_2019.And(builtup_2024.Not()).rename('change')

def frac(image, band):
    return image.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nairobi, scale=10, maxPixels=1e9
    ).getInfo()[band] * 100

frac_2019 = frac(builtup_2019, 'builtup')
frac_2024 = frac(builtup_2024, 'builtup')
frac_new = frac(new_builtup, 'change')
frac_lost = frac(lost_builtup, 'change')

print(f'Built-up fraction, 2019 (normalized): {frac_2019:.1f}%')
print(f'Built-up fraction, 2024: {frac_2024:.1f}%')
print(f'Net change: {frac_2024 - frac_2019:+.1f} percentage points')
print(f'New built-up (2019→2024 growth): {frac_new:.1f}% of Nairobi')
print(f'Lost built-up (noise check, should be small): {frac_lost:.1f}% of Nairobi')

Built-up fraction, 2019 (normalized): 40.2%
Built-up fraction, 2024: 37.2%
Net change: -3.0 percentage points
New built-up (2019→2024 growth): 8.2% of Nairobi
Lost built-up (noise check, should be small): 11.0% of Nairobi


## Visualize

In [4]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}
change_vis = {'min': 0, 'max': 1, 'palette': ['black', 'yellow']}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(composite_2024, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, '2024 true color', False)
Map.addLayer(builtup_2019, builtup_vis, '2019 built-up (normalized)', False)
Map.addLayer(builtup_2024, builtup_vis, '2024 built-up')
Map.addLayer(new_builtup, change_vis, 'New built-up (2019→2024)')
Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [5]:
for name, image in [
    ('builtup_2019', builtup_2019),
    ('builtup_2024', builtup_2024),
    ('new_builtup_2019_2024', new_builtup),
]:
    vis = change_vis if name.startswith('new') else builtup_vis
    url = image.getThumbURL({**vis, 'region': nairobi, 'dimensions': 800})
    path = f'../data/processed/nairobi_{name}.png'
    urllib.request.urlretrieve(url, path)
    print(f'Saved {path}')

Saved ../data/processed/nairobi_builtup_2019.png


Saved ../data/processed/nairobi_builtup_2024.png


Saved ../data/processed/nairobi_new_builtup_2019_2024.png


## Summary

| Metric | Value |
|---|---|
| Built-up fraction, 2019 (normalized) | 40.2% |
| Built-up fraction, 2024 | 37.2% |
| Net change | -3.0 percentage points |
| New built-up (growth) | 8.2% of Nairobi |
| Lost built-up (noise check) | 11.0% of Nairobi |

**Honest state of this result:** cloud/shadow masking (fixed in `acquisition.py`, benefits every
notebook downstream) plus mean/stdDev band normalization together brought the "lost built-up"
noise-check down from the original 32.1% to 11.0% — substantial progress, but not zero, and the
visual output shows why: both years' built-up maps are believably speckled (matching known
geography — dense CBD core, National Park correctly dark in both), but the new-built-up layer is
diffuse salt-and-pepper noise scattered evenly across the whole city rather than concentrated
growth at the urban fringe, which is what real 5-year urban growth should look like spatially.
That pattern is the signature of residual per-pixel classification instability, not genuine
change — most likely non-linear or spatially non-uniform radiometric differences between the two
composites (differential atmospheric haze, antecedent soil moisture) that one global linear
correction can't fully remove. A ~40x40 white patch in the northwest of the 2019 map is a
still-masked, heavily cloud-contaminated patch the SCL mask couldn't recover data for.

**Net -3.0pp is not treated as a genuine finding.** It shows Nairobi's built-up fraction
*shrinking* 2019→2024, which is implausible on its face for a growing city, and sits well within
the residual noise band established by the "lost built-up" sanity check. The honest conclusion
of this notebook is methodological, not substantive: **naive independent per-date classification
with an absolute-reflectance RF is not yet reliable enough for pixel-level change detection here,
even after masking and normalization** — the aggregate fraction numbers are closer to trustworthy
than the pixel-level change map, but neither should be reported as a clean growth measurement
without a better normalization scheme.

**Recommended follow-up:** pseudo-invariant-feature (PIF) based normalization using a curated set
of known-stable calibration targets was tried using Nairobi National Park's interior as the sole
PIF region — it overcorrected (2019 built-up fraction collapsed to 5.3%, clearly wrong) because
that region's own 2019 statistics were still cloud-contaminated pre-fix, and wasn't re-tested
post-masking. A more careful PIF set (e.g. several small, verified-stable regions, or a proper
atmospheric correction step) is the most promising next step if pixel-level change detection is
needed — out of scope for this pass. Recorded as a limitation, not silently hidden.